# How does Phi-3 Mini Perform on the SAT?

## Importing the Model

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/Phi-3-mini-4k-instruct" # This is the model we are using
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)
model.generation_config.pad_token_id = tokenizer.pad_token_id

tokenizer_config.json: 0.00B [00:00, ?B/s]

/opt/conda/lib/python3.12/site-packages/huggingface_hub/file_download.py:799: UserWarning: Not enough free disk space to download the file. The expected file size is: 0.50 MB. The target location /home/jovyan/.cache/huggingface/hub/models--microsoft--Phi-3-mini-4k-instruct/blobs only has 0.39 MB free disk space.
  warnings.warn(


tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

OSError: Can't load tokenizer for 'microsoft/Phi-3-mini-4k-instruct'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/Phi-3-mini-4k-instruct' is the correct path to a directory containing all relevant files for a LlamaTokenizerFast tokenizer.

### Checkpoint #1
Let's take a step back and think critically about the model we chose to upload. Use these resources to help you answer the following question:

Article on Phi-3 Mini
https://huggingface.co/microsoft/Phi-3-mini-4k-instruct

Microsoft's Overview of Phi-3 Models
https://azure.microsoft.com/en-us/blog/introducing-phi-3-redefining-whats-possible-with-slms/

In [ ]:
# RUN THIS CELL TO ANSWER CHECKPOINT #1
user_response = input("Why do you think we imported Phi-3-mini compared to other Phi-3 models?\n")
file_path = 'answers.txt'
with open(file_path, 'w') as file:
    file.write(user_response)
    file.write('\n') 

In [ ]:
def ask_sat_question(question, options):
    # Construct the full prompt including the question, options, and instructions for the model
    prompt = f"""
    Please reason step by step, and put your final answer within \\boxed{{}}.
    Here is an SAT-style multiple-choice question:
    
    Question: {question}
    Options:
    {options}

    Please provide your detailed reasoning and select the single best answer.
    """
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    
    # Format the prompt into the required chat template for Phi-3
    messages = [
        {"role": "system", "content": "You are a helpful assistant that excels at academic questions."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate the response
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens = 512, 
        do_sample = True,
        temperature = 0.6,
        top_p = 0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return response

This is an example of how we would ask Phi-3 to answer a sample question for us. Run it to see how Phi-3 responds!

In [ ]:
question_text = "If 3x + 5 = 14, what is the value of x?"
options_text = "A) 2\nB) 3\nC) 4\nD) 5"

response = ask_sat_question(question_text, options_text)
print(response)

### Using the SAT Questionbank

We can use a SAT Questionbank (linked here: https://pinesat.com/api/questions) to pick certain questions based on difficulty or domain. This can let us compare how the model performs depending on the type of "thinking" required.

The difficulty levels are "Easy", "Medium", or "Hard".

The available subjects in the question bank are...

    "Information and Ideas" — reading comprehension, main ideas
    "Standard English Conventions" — grammar, punctuation, sentence structure
    "Expression of Ideas" — rhetoric, author's purpose, transitions
    "Craft and Structure" — literary techniques, vocabulary in context

Take a look at the JSON file to get a sense of how the questionbank is structured: https://pinesat.com/api/questions

### Checkpoint #2

In [ ]:
# RUN THIS CELL TO ANSWER CHECKPOINT #2
user_response = input("Why do you think we are testing the model with these specific domains? Why not math?\n")
file_path = 'answers.txt'
with open(file_path, 'a') as file:
    file.write(user_response)
    file.write('\n') 

First, we will create a function to read the JSON file and fetch a random question for us, depending on the difficulty level and subject area.

In [ ]:
import random, requests

def get_sat_question(difficulty=None, subject=None):
    questions = requests.get("https://pinesat.com/api/questions").json()
    if difficulty: questions = [q for q in questions if q["difficulty"].lower() == difficulty.lower()]
    if subject: questions = [q for q in questions if subject.lower() in q["domain"].lower()]
    return random.choice(questions) if questions else None

Here, we didn't specify a difficulty or domain subject. Take this as an example of how to use this function.

In [ ]:
q = get_sat_question()
question = q["question"]["question"]
choices = q["question"]["choices"]
passage = q["question"]["paragraph"]
answer = q["question"]["correct_answer"]

# Some questions have a passage associated with them for the testtaker to read before answering the question.
# If that's the case, let's use that as the question instead
if passage and passage != "null":
    question = passage

In [ ]:
question

In [ ]:
choices

In [ ]:
passage

In [ ]:
question_text = question
options_text = choices

response = ask_sat_question(question_text, options_text)
print(response)

In [ ]:
answer

### Checkpoint #3

Now, you try! Write some code to pick a random Easy level question and have Phi-3 attempt the question. Did Phi-3 get the question right?

In [ ]:
### YOUR CODE HERE

question_text = ...
options_text = ...
response = ask_sat_question(question_text, options_text)
print(response)

In [ ]:
## Did Phi-3 get the question right? How can you tell? Write the code below.

### YOUR CODE HERE

Let's scale up and create a function to check if Phi-3 got the right answer automatically. This way, we can run several questions and see how Phi-3 performs overall.

In [ ]:
import re
def extract_answer(response, correct_answer):
    pattern = r"/correct\sanswer\sis\s(.+)\."
    a = re.search(pattern, correct_answer)
    if correct_answer in response:
        return 'CORRECT'
    else:
        return 'INCORRECT'

In [ ]:
extract_answer(response, answer)

### 12-Question Performance: Phi-3 Mini

In [ ]:
q1 = get_sat_question('Easy', 'Information and Ideas')
q2 = get_sat_question('Easy', 'Standard English Conventions')
q3 = get_sat_question('Easy', 'Expression of Ideas')
q4 = get_sat_question('Easy', 'Craft and Structure')

q5 = get_sat_question('Medium', 'Information and Ideas')
q6 = get_sat_question('Medium', 'Standard English Conventions')
q7 = get_sat_question('Medium', 'Expression of Ideas')
q8 = get_sat_question('Medium', 'Craft and Structure')

q9 = get_sat_question('Hard', 'Information and Ideas')
q10 = get_sat_question('Hard', 'Standard English Conventions')
q11 = get_sat_question('Hard', 'Expression of Ideas')
q12 = get_sat_question('Hard', 'Craft and Structure')

question_bank = [q1, q2, q3, q4, q5, q6, q7, q8, q9, q10, q11, q12]

In [ ]:
import pandas as pd
df = pd.DataFrame(columns = ['Domain', 'Difficulty', 'Question', 'Choices', 'Response', 'Correct?'])
df

In [ ]:
for q in question_bank:
    question = q["question"]["question"]
    choices = q["question"]["choices"]
    passage = q["question"]["paragraph"]
    answer = q["question"]["correct_answer"]
    domain = q["domain"]
    difficulty = q['difficulty']
    if passage and passage != "null":
        question = passage

    response = ask_sat_question(question, choices)
    check = extract_answer(response, answer)

    new_row = {'Domain': domain, 'Difficulty': difficulty, 'Question': question, 'Choices': choices, 'Response': response, 'Correct?': check}
    df.loc[len(df)] = new_row

In [ ]:
df

In [ ]:
df.to_excel('Phi3Mini_results.xlsx')